# Credit Score Ranking — The 12 Step Method

**Question:** Is this customer Poor, Average or Good?  (Answer: Poor, Average or Good)

**Dataset:** `credit.csv` — keep this file in the **same folder** as this notebook.

**Model we will use:** Logistic Regression

---

### The 12 Steps

| Step | What we do |
|---|---|
| 0 | Install libraries (only once on a new computer) |
| 1 | Import libraries |
| 2 | Import data |
| 3 | Copy data into a DataFrame |
| 4 | Find nulls, outliers, skew and bias |
| 5 | Scaling the data |
| 6 | Fit and transform |
| 7 | Divide into train and test |
| 8 | Train the model (`model.fit`) |
| 9 | Make predictions (`model.predict`) |
| 10 | Check the model on test data |
| 11 | Test with our own new values |
| 12 | Export the model for the web app |

**Every project follows these same 12 steps.** Only the data and the model change.

Run each cell with **Shift + Enter**.

---
# STEP 0 — Install Libraries

Run this cell **only once** on a new computer. After that you can skip it.

If a library is already installed, Python will simply say *"Requirement already satisfied"* —
that is not an error.

In [ ]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn
!pip install scikit-learn
!pip install joblib

**Notes**

- The `!` at the start tells Jupyter to run the command outside Python.
- The library is called **scikit-learn** when installing, but **sklearn** when importing.
- If you are using Anaconda, most of these are already installed.
- After installing, restart the kernel once: menu **Kernel → Restart**.

---
# STEP 1 — Import Libraries

These are the tools we need. We import them once at the top.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Step 1 done - all libraries imported")

---
# STEP 2 — Import Data

We read the CSV file directly from the disk.

The file `credit.csv` must be in the **same folder** as this notebook.

In [ ]:
data = pd.read_csv("credit.csv")

print("Step 2 done - data imported")
print("Rows and Columns:", data.shape)

If your file is somewhere else, write the full path instead:

```python
data = pd.read_csv("C:/AI_Training/credit.csv")        # Windows
data = pd.read_csv("/home/user/AI_Training/credit.csv")  # Mac / Linux
```

In Windows always use forward slashes `/` in the path.

---
# STEP 3 — Copy Data into a DataFrame

We make a copy and work on the copy.

The original `data` stays safe, so if we make a mistake we can start again without reading
the file.

In [ ]:
df = data.copy()

df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

**Columns**

| Column | Meaning |
|---|---|
| age | customer's age |
| monthly_income | income per month |
| loan_amount | how much they want to borrow |
| existing_loans | loans they already have |
| missed_payments | payments missed in the past |
| credit_history_years | how long they have used credit |
| credit_score | **Poor / Average / Good — this is our answer (target)** |

---
# STEP 4 — Find Nulls, Outliers, Skew and Bias

## 4.1 Nulls (missing values)

In [ ]:
df.isnull().sum()

We fill the missing values with the **median** of that column.

We use median (the middle value) because it is not affected by very large wrong values.

In [ ]:
df["age"] = df["age"].fillna(df["age"].median())
df["monthly_income"] = df["monthly_income"].fillna(df["monthly_income"].median())
df["loan_amount"] = df["loan_amount"].fillna(df["loan_amount"].median())
df["existing_loans"] = df["existing_loans"].fillna(df["existing_loans"].median())
df["missed_payments"] = df["missed_payments"].fillna(df["missed_payments"].median())
df["credit_history_years"] = df["credit_history_years"].fillna(df["credit_history_years"].median())

print("Nulls now:", df.isnull().sum().sum())

## 4.2 Outliers

An outlier is a value that is far away from all the others — for example a humidity of 999%
or an age of 200 years.

A **boxplot** shows them. Every dot outside the box is an outlier.

In [ ]:
df.plot(kind="box", subplots=True, layout=(1, 6), figsize=(16, 4))
plt.show()

Now we remove the impossible values.

We know the correct range for each column, so we simply keep only the sensible rows.

In [ ]:
print("Rows before:", df.shape[0])

df = df[df["age"] <= 100]
df = df[df["monthly_income"] <= 1000000]

print("Rows after :", df.shape[0])

In [ ]:
df.plot(kind="box", subplots=True, layout=(1, 6), figsize=(16, 4))
plt.show()

## 4.3 Skew

Skew tells us if the data is balanced or leaning to one side.

| Skew value | Meaning |
|---|---|
| between -0.5 and +0.5 | good, balanced |
| between ±0.5 and ±1 | slightly leaning |
| more than +1 or less than -1 | strongly leaning, should be fixed |

In [ ]:
df.skew(numeric_only=True).round(2)

### What does skew look like?

The numbers above are easier to understand as pictures.

In [ ]:
left_skew  = np.random.beta(8, 2, 5000)      # tail on the LEFT
normal     = np.random.normal(0, 1, 5000)     # balanced, bell shape
right_skew = np.random.beta(2, 8, 5000)       # tail on the RIGHT

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.hist(left_skew, bins=40, color="tomato")
plt.title("Negative skew - long tail on the LEFT")

plt.subplot(1, 3, 2)
plt.hist(normal, bins=40, color="mediumseagreen")
plt.title("No skew - balanced bell shape - BEST")

plt.subplot(1, 3, 3)
plt.hist(right_skew, bins=40, color="cornflowerblue")
plt.title("Positive skew - long tail on the RIGHT")

plt.show()

### Now the shape of our own columns

In [ ]:
df.hist(figsize=(15, 8), bins=30, color="steelblue", edgecolor="black")
plt.suptitle("Shape of every column")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
df.skew(numeric_only=True).plot(kind="bar", color="darkorange")
plt.axhline(1, color="red", linestyle="--")
plt.axhline(-1, color="red", linestyle="--")
plt.title("Skew of each column (inside the red lines = fine)")
plt.ylabel("skew value")
plt.show()

### How do we FIX skewed data?

If a column is strongly skewed, we squeeze the long tail using a maths function.
The three common fixes are:

| Fix | Code | Use when |
|---|---|---|
| Log transform | `np.log(df["col"] + 1)` | strong positive skew (money, income, sales) |
| Square root | `np.sqrt(df["col"])` | mild positive skew (counts) |
| Square | `df["col"] ** 2` | negative skew |

We add `+ 1` inside the log because `log(0)` does not exist.

**Log is the most used one.** It pulls very large values closer to the rest, without
changing their order.

Our column **`monthly_income`** is strongly skewed, so we fix it now with a log transform.

In [ ]:
print("Skew BEFORE fix:", round(df["monthly_income"].skew(), 2))

df["monthly_income"] = np.log(df["monthly_income"] + 1)

print("Skew AFTER fix :", round(df["monthly_income"].skew(), 2))

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(data["monthly_income"].dropna(), bins=40, color="tomato")
plt.title("BEFORE - long tail on the right")

plt.subplot(1, 2, 2)
plt.hist(df["monthly_income"], bins=40, color="mediumseagreen")
plt.title("AFTER log transform - balanced")

plt.show()

The shape is now balanced.

**Remember this:** because we changed `monthly_income` here, the web app must do exactly the same
thing to any new value before predicting. We will handle that in Step 11.

## 4.4 Bias in the data

A dataset is **biased** (also called *imbalanced*) when one answer appears far more often
than the other.

This is dangerous. If 95% of rows say No, a lazy model can answer No every single time and
still be 95% "accurate" — while being completely useless.

Let us check our answer column.

In [ ]:
print(df["credit_score"].value_counts())
print()
print((df["credit_score"].value_counts(normalize=True) * 100).round(1))

In [ ]:
sns.countplot(x="credit_score", data=df)
plt.title("How many of each answer do we have?")
plt.show()

### How do we FIX biased data?

| Fix | How | Note |
|---|---|---|
| Collect more data | get more rows of the rare answer | always the best fix |
| `class_weight="balanced"` | one word inside the model | easiest, we use this |
| Oversampling (SMOTE) | create copies of the rare rows | needs an extra library |
| Undersampling | delete rows of the common answer | throws away data |
| Look at **recall**, not accuracy | change how you judge the model | always do this |

**Our data is reasonably balanced**, so no fix is needed here.

Always run this check anyway — it takes 5 seconds and it saves you from a useless model.

---
# STEP 5 — Scaling the Data

## 5.1 Separate X and y

- **X** = the input columns
- **y** = the answer column

In [ ]:
X = df.drop("credit_score", axis=1)
y = df["credit_score"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

## 5.2 Why do we need scaling?

Look at the numbers below. Some columns have very big values and some have very small ones.

The model can think the bigger number is more important, which is wrong.
Scaling brings every column to the same range.

In [ ]:
X.describe().round(2)

## 5.3 Create the scaler

`StandardScaler` changes every column so that its mean becomes 0.

In [ ]:
scaler = StandardScaler()

print("Step 5 done - scaler is ready")

---
# STEP 6 — Fit and Transform

- **fit** = the scaler looks at the data and learns the mean and the spread
- **transform** = the scaler changes the numbers

`fit_transform` does both together in one line.

In [ ]:
X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

X_scaled.head()

In [ ]:
X_scaled.describe().round(2)

Look at the `mean` row above — every column is now close to **0**.

The shape of the data did not change, only the scale.

---
# STEP 7 — Divide into Train and Test

We keep **80% for training** and **hide 20% for testing**.

The model learns from the training part only. The testing part is the exam — the model has
never seen it. This is the only honest way to check a model.

`stratify=y` keeps the same mix of answers in both parts.
`random_state=42` makes everyone in the class get the same split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)

print("Training rows:", X_train.shape[0])
print("Testing rows :", X_test.shape[0])

---
# STEP 8 — Train the Model

This is the actual machine learning. It is **one line**: `model.fit()`

## Why Logistic Regression?

**Logistic Regression** also works when there are **three** answers instead of two.

It calculates a chance for Poor, a chance for Average and a chance for Good, then picks the
highest one. The three chances always add up to 100%.

`max_iter=2000` simply gives the model more time to find the best answer.

In [ ]:
model = LogisticRegression(max_iter=2000)

model.fit(X_train, y_train)

print("Step 8 done - model is trained")

---
# STEP 9 — Make Predictions

Now we give the model the **test data** (which it has never seen) and ask for its answers.

In [ ]:
y_pred = model.predict(X_test)

print("First 10 predictions :", list(y_pred[:10]))
print("First 10 real answers:", list(y_test[:10]))

In [ ]:
result = pd.DataFrame({"Real Answer": y_test.values,
                       "Model Prediction": y_pred})

result.head(15)

---
# STEP 10 — Check the Model

## 10.1 How many did the model get correct?

In [ ]:
correct = (y_pred == y_test).sum()
total = len(y_test)

print("Total test rows      :", total)
print("Correctly predicted  :", correct)
print("Wrongly predicted    :", total - correct)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy in percent:", round(accuracy * 100, 2), "%")

In [ ]:
baseline = df["credit_score"].value_counts(normalize=True).max()

print("If we always answered the most common answer:", round(baseline * 100, 2), "%")
print("Our model                                   :", round(accuracy * 100, 2), "%")
print()
print("Our model must be clearly better than the baseline, otherwise it learned nothing.")

## 10.2 Confusion Matrix

This table shows exactly which mistakes the model made.
The numbers on the diagonal are the correct answers.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

print(cm)

In [ ]:
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted " + c for c in model.classes_],
            yticklabels=["Actual " + c for c in model.classes_])
plt.title("Confusion Matrix")
plt.show()

## 10.3 Full report

In [ ]:
print(classification_report(y_test, y_pred))

| Word | Meaning |
|---|---|
| precision | when the model gave this answer, how often was it right |
| recall | out of all the real cases, how many did the model catch |
| f1-score | balance of precision and recall |
| support | how many rows of that type were in the test data |

## 10.4 Check the training score also

If the training score is very high but the test score is low, the model has **memorised**
the training data instead of learning. That is called **overfitting**.

In [ ]:
train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)

print("Training accuracy:", round(train_accuracy * 100, 2), "%")
print("Testing accuracy :", round(test_accuracy * 100, 2), "%")
print("Difference       :", round((train_accuracy - test_accuracy) * 100, 2), "%")

A small difference means the model is healthy.

A large difference (more than about 10%) means overfitting — use a simpler model, or a
smaller `max_depth`, or collect more data.

---
# STEP 11 — Test the Model with New Data

Now we give the model values that are **not in the file at all** — values that we type
ourselves.

**Very important:** new data must be prepared in exactly the same way as the training data.

1. Put the values in a DataFrame with the **same column names and same order**
2. Scale it with the **same scaler**
3. Only then call `predict`

Column order: `age, monthly_income, loan_amount, existing_loans, missed_payments, credit_history_years`

## 11.1 Type your own values

Change the numbers below to anything you like, then run the cell.

In [ ]:
new_row = pd.DataFrame([[35, 90000, 150000, 0, 0, 12]], columns=X.columns)

new_row

**Do not forget the log transform.**

In Step 4.3 we changed `monthly_income` with a log. The model learned from log values, so any new
value must also be converted the same way, otherwise the answer will be completely wrong.

In [ ]:
new_row["monthly_income"] = np.log(new_row["monthly_income"] + 1)

new_row

## 11.2 Scale the new data

We use `transform` here, **not** `fit_transform`.

The scaler already learned in Step 6. If we used `fit_transform` here, it would learn again
from one single row and the prediction would be wrong.

In [ ]:
new_row_scaled = scaler.transform(new_row)

new_row_scaled = pd.DataFrame(new_row_scaled, columns=X.columns)

new_row_scaled

## 11.3 Predict

In [ ]:
prediction = model.predict(new_row_scaled)

print("Is this customer Poor, Average or Good?", prediction[0])

## 11.4 How sure is the model?

`predict` gives only the answer. `predict_proba` gives the **chance** behind that answer.

In [ ]:
probability = model.predict_proba(new_row_scaled)

chances = pd.DataFrame({"answer": model.classes_,
                        "chance_%": (probability[0] * 100).round(2)})

chances

## 11.5 Try a few different cases at once

In [ ]:
many_rows = pd.DataFrame([
    [35, 90000, 150000, 0, 0, 12],    # strong customer
    [24, 20000, 400000, 3, 6, 1],    # risky customer
    [40, 50000, 200000, 1, 2, 8],    # average customer
], columns=X.columns)
many_rows["monthly_income"] = np.log(many_rows["monthly_income"] + 1)

many_rows_scaled = pd.DataFrame(scaler.transform(many_rows), columns=X.columns)

results = many_rows.copy()
results["prediction"] = model.predict(many_rows_scaled)

results

---
# STEP 12 — Export the Model for the Web App

Training took a few seconds here, but on real data it can take hours. We do not want to
train the model again every time someone opens the app.

So we **save the trained model into a file**. This is called **pickling**.

Think of it like cooking: training is the cooking, the `.pkl` file is the meal in the
freezer, and the app just reheats it.

## 12.1 What do we save?

We must save **three** things together, not just the model:

| What | Why |
|---|---|
| the **model** | it makes the prediction |
| the **scaler** | new data must be scaled the same way, or the answer will be wrong |
| the **column names** | so the app sends the columns in the correct order |

In [ ]:
import joblib

package = {
    "model": model,
    "scaler": scaler,
    "columns": list(X.columns)
}

joblib.dump(package, "credit_model.pkl")

print("Model saved as credit_model.pkl")

## 12.2 Always test the saved file

Never build an app on top of a `.pkl` file you have not tested.

In [ ]:
loaded = joblib.load("credit_model.pkl")

loaded_model = loaded["model"]
loaded_scaler = loaded["scaler"]
loaded_columns = loaded["columns"]

print("Columns the model expects:", loaded_columns)

In [ ]:
test_row = pd.DataFrame([[35, 90000, 150000, 0, 0, 12]], columns=loaded_columns)
test_row["monthly_income"] = np.log(test_row["monthly_income"] + 1)

test_row_scaled = pd.DataFrame(loaded_scaler.transform(test_row), columns=loaded_columns)

print("Prediction from the saved file:", loaded_model.predict(test_row_scaled)[0])

If the answer above is the same as in Step 11.3, the file is correct and ready to use.

## 12.3 Create requirements.txt automatically

When we put the app on the internet, the server is an empty computer. It has no pandas and
no scikit-learn until we tell it what to install. That list goes in **requirements.txt**.

There is one important detail. A `.pkl` file saved by scikit-learn version 1.5 may **refuse
to open** on a server that installed version 1.8. This is the most common reason a working
app fails after hosting.

The safe way is to write down the exact versions of **this** computer, because this is the
computer that created the `.pkl` file.

In [ ]:
import sklearn

lines = [
    "streamlit",
    "pandas==" + pd.__version__,
    "numpy==" + np.__version__,
    "scikit-learn==" + sklearn.__version__,
    "joblib==" + joblib.__version__,
]

text = "\n".join(lines)

file = open("requirements.txt", "w")
file.write(text)
file.close()

print("requirements.txt created with these lines:")
print()
print(text)

## 12.4 What to upload to GitHub

| File | Needed for the app? |
|---|---|
| `credit_app.py` | YES - the app itself |
| `credit_model.pkl` | YES - the trained model |
| `requirements.txt` | YES - the list of libraries |
| `credit.csv` | No - only used for training |
| this notebook | No - but nice to keep |

Upload the **first three** to a **public** GitHub repository, then deploy from
share.streamlit.io with Main file path = `credit_app.py`.

The app does **not** read the CSV file. It only reads the `.pkl` file. That is the whole
point of exporting the model.

---
# Finished

| Step | What we did |
|---|---|
| 0 | Installed the libraries |
| 1 | Imported libraries |
| 2 | Imported the CSV file |
| 3 | Copied it into a DataFrame |
| 4 | Fixed nulls, removed outliers, checked skew and bias |
| 5 | Separated X and y, created the scaler |
| 6 | fit_transform on the data |
| 7 | Split into 80% train and 20% test |
| 8 | Trained the Logistic Regression model |
| 9 | Predicted on the test data |
| 10 | Checked how many predictions were correct |
| 11 | Tested with our own new values |
| 12 | Saved `credit_model.pkl` and created requirements.txt |

**Next:** run the web app.

```
streamlit run credit_app.py
```